# 01 — Wikipedia Dataset Preprocessing

**Pipeline:** BZ2 Wikipedia dump → MinHash-deduped JSONL shards → HuggingFace Dataset Hub

```
Wikipedia dump (.xml.bz2)
        │
        ▼
  WikiExtractor (text extraction)
        │
        ▼
  Tokenization + Filtering
        │  remove stubs < 100 tokens
        ▼
  MinHash Deduplication
        │  LSH threshold = 0.85
        ▼
  JSONL Sharding (1 GB each)
        │
        ▼
  HuggingFace Dataset Hub push
```

**References:**
- C4 / mC4: Raffel et al. (2020) https://arxiv.org/abs/1910.10683
- The Pile: Gao et al. (2020) https://arxiv.org/abs/2101.00027

> **CPU-friendly:** This notebook runs on CPU only — no GPU required.

In [ ]:
from __future__ import annotations

import bz2
import hashlib
import json
import logging
import os
import re
import struct
import xml.etree.ElementTree as ET
from collections import defaultdict
from pathlib import Path

import wandb

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CFG = {
    "input_dump": "data/raw/enwiki-latest-pages-articles.xml.bz2",
    "output_dir": "data/processed/wiki-jsonl",
    "shard_size_mb": 1024,
    "min_tokens": 100,
    "dedup_threshold": 0.85,
    "hf_dataset_name": "TylrDn/wiki-jsonl",
}

# wandb tracking (set WANDB_DISABLED=true for offline runs)
if not os.getenv("WANDB_DISABLED"):
    wandb.init(project="ai-transformer-research-hub", config=CFG, job_type="preprocess")

log.info("Config: %s", CFG)


In [ ]:
# Stage 1 — WikiExtractor: BZ2 dump → clean article text
# WikiExtractor approach: Attardi (2015) https://github.com/attardi/wikiextractor
# C4/mC4 pipeline: Raffel et al. (2020) https://arxiv.org/abs/1910.10683

_MARKUP_PATTERNS = [
    (re.compile(r"\{\{[^}]*\}\}", re.DOTALL), ""),
    (re.compile(r"\{\|.*?\|\}", re.DOTALL), ""),
    (re.compile(r"\[\[(File|Image):[^\]]*\]\]", re.I), ""),
    (re.compile(r"\[\[[^\]|]*\|([^\]]*)\]\]"), r"\1"),
    (re.compile(r"\[\[([^\]]*)\]\]"), r"\1"),
    (re.compile(r"\[https?://\S+\s+([^\]]+)\]"), r"\1"),
    (re.compile(r"\[https?://\S+\]"), ""),
    (re.compile(r"<[^>]+>"), ""),
    (re.compile(r"={2,}[^=]+=={2,}"), ""),
    (re.compile(r"\s+"), " "),
]


def clean_wikitext(raw: str) -> str:
    """Strip MediaWiki markup and return plain prose."""
    text = raw
    for pattern, repl in _MARKUP_PATTERNS:
        text = pattern.sub(repl, text)
    return text.strip()


def extract_articles(dump_path: Path, max_articles: int = 0) -> list[dict]:
    """Stream-parse a Wikipedia XML dump and return clean article dicts.

    Falls back gracefully when the dump is absent so the notebook can be
    exercised in dry-run / CI environments.

    Args:
        dump_path: Path to the .xml.bz2 (or .xml) dump file.
        max_articles: Stop after this many articles; 0 means no limit.

    Returns:
        List of dicts with keys: id (str), title (str), text (str), meta (dict).
    """
    articles: list[dict] = []
    ns = "{http://www.mediawiki.org/xml/export-0.10/}"

    if not dump_path.exists():
        log.warning(
            "Dump not found: %s — running in dry-run mode (returning empty list)",
            dump_path,
        )
        return articles

    opener = bz2.open if str(dump_path).endswith(".bz2") else open
    with opener(dump_path, "rb") as fh:  # type: ignore[call-overload]
        for _event, elem in ET.iterparse(fh, events=("end",)):
            if elem.tag != f"{ns}page":
                continue
            if (elem.findtext(f"{ns}ns") or "0") != "0":
                elem.clear()
                continue
            title = elem.findtext(f"{ns}title") or ""
            text_el = elem.find(f".//{ns}text")
            raw = (text_el.text or "") if text_el is not None else ""
            articles.append(
                {"id": str(len(articles)), "title": title, "text": clean_wikitext(raw), "meta": {}}
            )
            elem.clear()
            if max_articles and len(articles) >= max_articles:
                break

    log.info("Extracted %d raw articles from %s", len(articles), dump_path)
    return articles


dump_path = Path(CFG["input_dump"])
articles = extract_articles(dump_path)
log.info("Stage 1 complete — %d articles", len(articles))


In [ ]:
# Stage 2 — Tokenization, filtering, MinHash deduplication
# MinHash LSH: Broder (1997) — near-duplicate detection via resemblance
# The Pile: Gao et al. (2020) https://arxiv.org/abs/2101.00027

_MERSENNE_PRIME = (1 << 61) - 1
_RNG = __import__("random").Random(42)
_HASH_PARAMS: list[tuple[int, int]] = [
    (_RNG.randint(1, _MERSENNE_PRIME), _RNG.randint(0, _MERSENNE_PRIME))
    for _ in range(128)
]
_NUM_PERM = len(_HASH_PARAMS)
_BANDS = 20
_ROWS = _NUM_PERM // _BANDS


def _minhash(text: str) -> list[int]:
    shingles = {text[i : i + 5] for i in range(max(len(text) - 4, 1))}
    sig = [_MERSENNE_PRIME] * _NUM_PERM
    for shingle in shingles:
        h = int(hashlib.md5(shingle.encode()).hexdigest(), 16)
        for j, (a, b) in enumerate(_HASH_PARAMS):
            sig[j] = min(sig[j], (a * h + b) % _MERSENNE_PRIME)
    return sig


def tokenize_and_filter(articles: list[dict], min_tokens: int) -> list[dict]:
    """Whitespace-tokenize and drop articles shorter than min_tokens."""
    kept: list[dict] = []
    for art in articles:
        tokens = art["text"].split()
        if len(tokens) < min_tokens:
            continue
        art["meta"]["token_count"] = len(tokens)
        kept.append(art)
    log.info(
        "Filtering: %d -> %d articles (min_tokens=%d)",
        len(articles), len(kept), min_tokens,
    )
    return kept


def minhash_dedup(articles: list[dict], threshold: float) -> list[dict]:
    """Remove near-duplicate articles with MinHash LSH (Jaccard >= threshold).

    MinHash LSH: Broder (1997) — On the resemblance and containment of documents.
    C4/mC4 deduplication: Raffel et al. (2020) https://arxiv.org/abs/1910.10683

    Args:
        articles: Tokenised, filtered article dicts.
        threshold: Jaccard similarity threshold for duplicate detection.

    Returns:
        Deduplicated list (first occurrence kept).
    """
    if not articles:
        return []

    sigs = [_minhash(a["text"]) for a in articles]
    buckets: dict[bytes, list[int]] = defaultdict(list)
    for idx, sig in enumerate(sigs):
        for band in range(_BANDS):
            key = bytes([band]) + struct.pack(
                f">{_ROWS}Q", *sig[band * _ROWS : (band + 1) * _ROWS]
            )
            buckets[key].append(idx)

    duplicates: set[int] = set()
    for group in buckets.values():
        if len(group) < 2:
            continue
        for i in range(len(group)):
            for j in range(i + 1, len(group)):
                a, b = group[i], group[j]
                if b in duplicates:
                    continue
                jaccard = sum(s1 == s2 for s1, s2 in zip(sigs[a], sigs[b])) / _NUM_PERM
                if jaccard >= threshold:
                    duplicates.add(b)

    deduped = [a for i, a in enumerate(articles) if i not in duplicates]
    log.info(
        "Dedup: removed %d/%d duplicates (threshold=%.2f)",
        len(duplicates), len(articles), threshold,
    )
    return deduped


filtered = tokenize_and_filter(articles, CFG["min_tokens"])
deduped = minhash_dedup(filtered, threshold=CFG["dedup_threshold"])
log.info("Stage 2 complete — %d articles after dedup", len(deduped))


In [ ]:
# Stage 3 — JSONL sharding  |  Stage 4 — HuggingFace Dataset Hub push
# Schema: {"id": str, "text": str, "meta": {...}}
# The Pile: Gao et al. (2020) https://arxiv.org/abs/2101.00027


def write_jsonl_shards(
    articles: list[dict],
    output_dir: Path,
    shard_size_mb: int,
) -> list[Path]:
    """Write articles to JSONL files capped at shard_size_mb each.

    Args:
        articles: Processed article dicts (id, title, text, meta).
        output_dir: Destination directory (created if absent).
        shard_size_mb: Approximate maximum size of each shard in megabytes.

    Returns:
        List of Path objects for every shard written.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    max_bytes = shard_size_mb * 1024 * 1024
    shard_paths: list[Path] = []
    shard_idx, current_size = 0, 0
    fh = None
    try:
        for art in articles:
            line = (
                json.dumps(
                    {"id": art["id"], "text": art["text"], "meta": art["meta"]},
                    ensure_ascii=False,
                )
                + "\n"
            )
            encoded = line.encode("utf-8")
            if fh is None or current_size + len(encoded) > max_bytes:
                if fh is not None:
                    fh.close()
                shard_path = output_dir / f"shard_{shard_idx:05d}.jsonl"
                fh = shard_path.open("w", encoding="utf-8")
                shard_paths.append(shard_path)
                shard_idx += 1
                current_size = 0
            fh.write(line)
            current_size += len(encoded)
    finally:
        if fh is not None:
            fh.close()
    log.info("Wrote %d JSONL shard(s) to %s", len(shard_paths), output_dir)
    return shard_paths


def push_to_hf_hub(shard_paths: list[Path], dataset_name: str) -> None:
    """Upload JSONL shards to HuggingFace Dataset Hub.

    Args:
        shard_paths: Local shard files to upload.
        dataset_name: HF dataset ID, e.g. 'TylrDn/wiki-jsonl'.
    """
    try:
        from datasets import load_dataset  # type: ignore

        if not shard_paths:
            log.warning("No shards to push — skipping HF Hub upload")
            return
        dataset = load_dataset(
            "json", data_files=[str(p) for p in shard_paths], split="train"
        )
        dataset.push_to_hub(dataset_name)
        log.info("Pushed %d examples -> %s", len(dataset), dataset_name)
    except Exception as exc:  # noqa: BLE001
        log.warning("HF Hub push skipped: %s", exc)


output_dir = Path(CFG["output_dir"])
shard_paths = write_jsonl_shards(deduped, output_dir, CFG["shard_size_mb"])
push_to_hf_hub(shard_paths, CFG["hf_dataset_name"])

total_tokens = sum(
    a["meta"].get("token_count", len(a["text"].split())) for a in deduped
)
dedup_ratio = 1.0 - len(deduped) / max(len(articles), 1)
log.info(
    "Pipeline complete — %d shards, %.1fM tokens, dedup_ratio=%.3f",
    len(shard_paths),
    total_tokens / 1e6,
    dedup_ratio,
)

if not os.getenv("WANDB_DISABLED"):
    wandb.log(
        {
            "shards_written": len(shard_paths),
            "total_tokens": total_tokens,
            "dedup_ratio": round(dedup_ratio, 4),
        }
    )
    wandb.finish()
